# День 11: сборка из моделей day10 (~25 мин) + короткое дообучение gru_full на тех же данных (~45 мин, по желанию)
Day10: обе модели, обученные ещё и на остатке валидации, стали лучше своих предшественников.

| модель | маска | все строки |
|---|---|---|
| ft2_swa (ft_swa + 3 эпохи на train и остатке валидации) | 0.66737 | 0.51227 |
| ft_swa | 0.66679 | 0.51096 |
| reg128m (GRU-128 с нуля на тех же данных) | 0.66718 | 0.51545 |
| reg128b (тот же рецепт, только train) | 0.66552 | 0.51174 |

Тройка ft2_swa + gru_full + reg128m с равными весами — 0.67115 против 0.67036 у отправленной. Ячейка оценки в day10 не успела закончиться (ноутбук сохранён, пока она работала), поэтому подбор весов и сборка не запускались.

**Часть A** (без обучения): оценка всех моделей, подбор весов для лучших пар и троек, сборка, проверка, скачивание.
**Часть B** (по желанию): gru_full — самая «непохожая» модель в смеси — продолжаем обучать 2 эпохи с маленьким шагом на train + остатке валидации (как ft2 для ft_swa). Второй zip — только если смесь с ним лучше результата части A хотя бы на 0.0003.

Перед запуском: **T4 GPU**; загрузите на Drive вместо `wunder/day2` папку `хакатон/day2` — изменился `ensemble_eval.py`.

## 1. GPU + Drive + код + numba как у скорера

In [1]:
!nvidia-smi --query-gpu=name --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')
W = '/content/drive/MyDrive/wunder'
!pip -q install "numba==0.67.0" onnx 2>&1 | grep -v "dependency resolver\|requires numba\|incompatible" || true
!rm -rf /content/day2 && cp -r $W/day2 /content/day2
%cd /content/day2
!grep -q "no-cv" ensemble_eval.py && echo "code: OK" || echo "OLD CODE on Drive - upload the new day2 folder"

Tesla T4
Mounted at /content/drive
/content/day2
code: OK


## 2. Валидация `valid_sub` (~1.5 мин) и общие функции

In [2]:
HELD = W + '/colab_data/valid_sub.parquet'
!python prepare_data.py --src $HELD --out /content/data/valid

import json, os, subprocess, sys
import numpy as np
sys.path.insert(0, '/content/day2')
from ensemble_eval import standardize, blend, wp, weight_grid
from common import SEQ_LEN, WARMUP
REF = W + '/submissions/gru128x2_v1'
PREDS = '/content/preds11'
Y = np.load('/content/data/valid/targets.npy')
M = np.load('/content/data/valid/is_scored.npy') & (np.arange(SEQ_LEN) >= WARMUP)
Ym = Y[M]

def sh(cmd):
    print('$', cmd if len(cmd) < 300 else cmd[:300] + ' ...', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

def swa(run, ks):
    return '+'.join(f'{run}/epoch_{k}.pt' for k in ks)

def evaluate(cands, report):
    found = {k: v for k, v in cands.items() if all(os.path.exists(p) for p in v.split('+'))}
    print('models:', list(found), '| missing:', [k for k in cands if k not in found])
    sh('python ensemble_eval.py --valid-dir /content/data/valid --out ' + PREDS + ' --no-cv --report ' + report + ' '
       + ' '.join(f'--model {k}={v}' for k, v in found.items()))
    return found, json.load(open(report))['equal_blends']

def best_blends(eb, found, must=None, top=4):
    """Best pairs/triples (time 0.53x / 0.8x): the top ones by equal weights, then weights tuned on a 0.1 grid."""
    keys = [k for v, k in sorted(((v, k) for k, v in eb.items() if k.count('+') <= 2), reverse=True)
            if must is None or must & set(k.split('+'))][:top]
    if not keys:
        return None, []
    Z = {n: standardize(np.load(f'{PREDS}/{n}.npy')[M]) for n in {n for k in keys for n in k.split('+')}}
    res = []
    for k in keys:
        names = k.split('+')
        grid = list(weight_grid(len(names), 0.1)) + [np.ones(len(names)) / len(names)]
        s, w = max((wp(Ym, blend([Z[n] for n in names], w)), tuple(float(x) for x in np.round(w, 3))) for w in grid)
        parts = [(n, x) for n, x in zip(names, w) if x > 0]
        res.append((s, parts))
        print(f'  {k:36s} equal {eb[k]:.5f} | weights {w} -> {s:.5f}')
    return max(res, key=lambda r: r[0])

def build(parts, found, name):
    out = W + '/submissions/' + name
    spec = ' '.join(f'--model "{n}={found[n]}@{x:.3f}"' for n, x in parts)
    if sh(f'python build_ensemble.py --valid-dir /content/data/valid --out {out} {spec}') != 0:
        raise SystemExit('build failed - see above')
    sh(f'python check_submission.py {out}.zip --data {HELD} --seqs 3 --rows 20000 --ref-dir {REF}')
    sh(f'python bench_speed.py {REF} {out} --rows 5000 --repeats 5')
    return out

100/600 seqs  17s  eta 83s
200/600 seqs  31s  eta 61s
300/600 seqs  44s  eta 44s
400/600 seqs  55s  eta 28s
500/600 seqs  69s  eta 14s
600/600 seqs  81s  eta 0s
{
 "src": "/content/drive/MyDrive/wunder/colab_data/valid_sub.parquet",
 "n_seqs": 600,
 "seq_len": 20000,
 "n_features": 112,
 "dtype": "float32",
 "has_is_scored": true,
 "nan_features": 0,
 "inf_features": 0,
 "nonfinite_targets": 0,
 "float16_overflow": 0,
 "scored_rows": 1144931
}


## 3. Часть A: оценка всех моделей, подбор весов, сборка (~20 мин)
Отправленная тройка ft_swa + gru_full + reg128 с равными весами — 0.67036 (лидерборд 0.65512). Сборка — если лучший вариант лучше неё хотя бы на 0.0003.

In [3]:
CANDS = {
    'ft_swa':   swa(W + '/runs/ft_baseline_full', range(2, 6)),
    'ft2_swa':  swa(W + '/runs/ft2', range(0, 3)),
    'ft2_e0':   W + '/runs/ft2/epoch_0.pt',
    'gru_full': W + '/runs/gru_full/best.pt',
    'reg128':   W + '/runs/reg128/best.pt',
    'reg128b':  W + '/runs/reg128b/best.pt',
    'reg128m':  W + '/runs/reg128m/best.pt',
}
FOUND, EB = evaluate(CANDS, W + '/day11a.json')
SENT = EB['ft_swa+gru_full+reg128']
A_S, A_PARTS = best_blends(EB, FOUND)
print(f'\nbest: {A_S:.5f} ({A_S - SENT:+.5f} vs sent)  ' + ' + '.join(f'{x:.2f}*{n}' for n, x in A_PARTS))
A_OUT = None
if A_S > SENT + 0.0003:
    A_OUT = build(A_PARTS, FOUND, 'ens11A_' + '_'.join(n for n, _ in A_PARTS))
    from google.colab import files
    files.download(A_OUT + '.zip')
else:
    print('nothing better than the sent ensemble')

models: ['ft_swa', 'ft2_swa', 'ft2_e0', 'gru_full', 'reg128', 'reg128b', 'reg128m'] | missing: []
$ python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds11 --no-cv --report /content/drive/MyDrive/wunder/day11a.json --model ft_swa=/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_2.pt+/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_3.pt+/content/drive/MyDriv ...
ft_swa         predicted (baseline, epoch 2, SWA of 4)
ft2_swa        predicted (baseline, epoch 0, SWA of 3)
ft2_e0         predicted (baseline, epoch 0)
gru_full       predicted (gru, epoch 5)
reg128         predicted (gru, epoch 6)
reg128b        predicted (gru, epoch 6)
reg128m        predicted (gru, epoch 6)

validation: 600 sequences, 1,144,931 scored rows. WP on the mask (and on all required rows):
  ft_swa         0.66679   (0.51096)
  ft2_swa        0.66737   (0.51227)
  ft2_e0         0.66736   (0.51221)
  gru_full       0.66370   (0.50515)
  reg128         0.66544   (0.50999)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Часть B (по желанию, ~45 мин): gru_full2 — gru_full + 2 эпохи на train и остатке валидации
Без регуляризации (у gru_full её не было), lr 1e-4. В начале лога `VALID (start ...)` должно быть ≈0.6637 (это gru_full).

In [4]:
import glob
URL = 'https://files.wundernn.io/wnn_connectome_starterpack.tar.gz'
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y aria2 > /dev/null 2>&1
!aria2c -c -x 16 -s 16 -k 1M --file-allocation=none --summary-interval=60 --console-log-level=warn -d /content -o pack.tar.gz $URL
!mkdir -p /content/sp && tar -xzf /content/pack.tar.gz -C /content/sp --wildcards '*datasets/train.parquet' '*datasets/valid.parquet' 2>&1 | grep -v LIBARCHIVE
!rm -f /content/pack.tar.gz
TRAIN_PQ = glob.glob('/content/sp/**/datasets/train.parquet', recursive=True)[0]
VALID_PQ = glob.glob('/content/sp/**/datasets/valid.parquet', recursive=True)[0]
GF = W + '/runs/gru_full/best.pt'
!python train.py --train-parquet $TRAIN_PQ --extra-parquet $VALID_PQ --exclude-seqs-from $HELD --valid-dir /content/data/valid \
    --out $W/runs/gru_full2 --arch gru --hidden 128 --init-from $GF --epochs 2 --batch 64 --chunk 1000 --lr 1e-4 \
    --keep-epochs --loader-workers 4

 *** Download Progress Summary as of Fri Nov 13 18:09:31 2026 *** 
=
[#6a82e4 12GiB/31GiB(38%) CN:16 DL:220MiB ETA:1m29s]
FILE: /content/pack.tar.gz
-

 *** Download Progress Summary as of Fri Nov 13 18:10:32 2026 *** 
=
[#6a82e4 24GiB/31GiB(76%) CN:16 DL:214MiB ETA:34s]
FILE: /content/pack.tar.gz
-


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
6a82e4|OK  |   197MiB/s|/content/pack.tar.gz

Status Legend:
(OK):download completed.
device=cuda amp=True Tesla T4
train seqs 11880 from /content/sp/wnn_connectome_starterpack/datasets/train.parquet + 1273 of 1873 sequences of /content/sp/wnn_connectome_starterpack/datasets/valid.parquet (without the 600 that are in valid_sub.parquet) (parquet, read on the fly by 4 threads); valid seqs 600
initialised from /content/drive/MyDrive/wunder/runs/gru_full/best.pt
VALID (start = initial weights) {"seqs": 600, "pred_std": [0.5740000009536743, 0.5683000087738037], "WP

## 5. Часть B: оценка с gru_full2 и сборка, если стало лучше (~15 мин)

In [5]:
CANDS.update({'gru_full2': W + '/runs/gru_full2/epoch_1.pt', 'gru_full2_swa': swa(W + '/runs/gru_full2', range(0, 2))})
FOUND, EB = evaluate(CANDS, W + '/day11b.json')
B_S, B_PARTS = best_blends(EB, FOUND, must={'gru_full2', 'gru_full2_swa'})
BASE = max(A_S, SENT)
if B_S is None:
    print('gru_full2 not found - run the training cell above first')
elif B_S > BASE + 0.0003:
    print(f'\nbest with gru_full2: {B_S:.5f} ({B_S - BASE:+.5f} vs part A)  ' + ' + '.join(f'{x:.2f}*{n}' for n, x in B_PARTS))
    B_OUT = build(B_PARTS, FOUND, 'ens11B_' + '_'.join(n for n, _ in B_PARTS))
    from google.colab import files
    files.download(B_OUT + '.zip')
else:
    print(f'\nbest with gru_full2: {B_S:.5f} ({B_S - BASE:+.5f} vs part A) - does not improve the blend, keep the zip from part A')

models: ['ft_swa', 'ft2_swa', 'ft2_e0', 'gru_full', 'reg128', 'reg128b', 'reg128m', 'gru_full2', 'gru_full2_swa'] | missing: []
$ python ensemble_eval.py --valid-dir /content/data/valid --out /content/preds11 --no-cv --report /content/drive/MyDrive/wunder/day11b.json --model ft_swa=/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_2.pt+/content/drive/MyDrive/wunder/runs/ft_baseline_full/epoch_3.pt+/content/drive/MyDriv ...
ft_swa         predictions loaded from /content/preds11/ft_swa.npy
ft2_swa        predictions loaded from /content/preds11/ft2_swa.npy
ft2_e0         predictions loaded from /content/preds11/ft2_e0.npy
gru_full       predictions loaded from /content/preds11/gru_full.npy
reg128         predictions loaded from /content/preds11/reg128.npy
reg128b        predictions loaded from /content/preds11/reg128b.npy
reg128m        predictions loaded from /content/preds11/reg128m.npy
gru_full2      predicted (gru, epoch 1)
gru_full2_swa  predicted (gru, epoch 0, SWA of 2)



## Что прислать
Вывод ячеек 3 и 5 (и лог обучения из 4). Zip из части A (и из части B, если собрался) можно отправлять, если в их проверке `RESULT: ready to submit`.